# Stage 2 — CREsted enhancer code analysis for 100 topic classes

This notebook is adapted to your human/macaque topic-classification models, following the CREsted enhancer code analysis tutorial:

1. Load AnnData and trained CREsted model.
2. Predict all regions and store predictions in `adata.layers`.
3. Build `combined = (adata.X + prediction) / 2`.
4. Select top specific regions per topic using `sort_and_filter_regions_on_specificity(..., method="gini")`.
5. Calculate class-specific contribution scores with `contribution_scores_specific(target_idx=None)`.
6. Run TF-MoDISco-lite with `crested.tl.modisco.tfmodisco`.

Inputs expected on Alvis:
- `runs/out/human_topics.h5ad`
- `runs/out/macaque_topics.h5ad`
- `runs/out/deeptopic_human/final_model.keras`
- `runs/out/deeptopic_macaque/final_model.keras`
- topic annotation/QC tables in `data/`

Outputs:
- `runs/out/stage2_crested_enhancer_code/<species>/...`


In [1]:
from __future__ import annotations

import os
from pathlib import Path

os.environ.setdefault("KERAS_BACKEND", "torch")

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42

import matplotlib.pyplot as plt

import anndata as ad
import crested
import keras


In [2]:
# =====================
# Config
# =====================
BASE = Path("/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt")
DATA = BASE / "data"
OUT = BASE / "runs" / "out"

OUTDIR = OUT / "stage2_crested_enhancer_code"
OUTDIR.mkdir(parents=True, exist_ok=True)

RUN_SPECIES = ["human"]  # start with human. Change to ["human", "macaque"] after human works.

TOP_K = 2000             # tutorial uses 2000. If too slow, use 500 or 1000.
SPECIFICITY_METHOD = "gini"
CONTRIB_METHOD = "integrated_grad"  # tutorial notes this is faster than expected integrated gradients.
MODEL_LAYER = "model_prediction"
COMBINED_LAYER = "combined"

RUN_CONTRIBUTION = True
RUN_TFMODISCO = True

# If TOMTOM is not available, keep REPORT=False.
REPORT = False
MAX_SEQLETS = 20000
MODISCO_WINDOW = 500      # your topic model uses 500 bp input; tutorial uses 1000 for its wider model.

DATASETS = {
    "human": {
        "adata": OUT / "human_topics.h5ad",
        "model": OUT / "deeptopic_human" / "final_model.keras",
        "genome_fasta": Path("/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/genomes/hg38/hg38.fa"),
        "chrom_sizes": Path("/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/genomes/hg38/hg38.chrom.sizes"),
        "annotation_candidates": DATA / "3_topic_annotation_human_pycistopic.tsv",
        "qc_candidates": DATA / "3_topic_qc_metrics_human.tsv",
    },
    "macaque": {
        "adata": OUT / "macaque_topics.h5ad",
        "model": OUT / "deeptopic_macaque" / "final_model.keras",
        "genome_fasta": Path("/mimer/NOBACKUP/groups/naiss2025-22-612/yiquan/macaque/rheMac10.fa"),
        "chrom_sizes": Path("/mimer/NOBACKUP/groups/naiss2025-22-612/yiquan/macaque/rheMac10.chrom.sizes"),
        "annotation_candidates": DATA / "3_topic_annotation_macaque_pycistopic.tsv",
        "qc_candidates": DATA / "3_topic_qc_metrics_macaque.tsv",
    },
}


In [3]:
def first_existing(candidates):
    if candidates is None:
        return None

    # allow single Path / string
    if isinstance(candidates, (str, Path)):
        candidates = [candidates]

    for p in candidates:
        if p is not None and Path(p).exists():
            return Path(p)

    return None


def normalize_topic_name(x):
    s = str(x)
    if s.lower().startswith("topic"):
        return "Topic" + s.replace("Topic", "").replace("topic", "")
    try:
        return f"Topic{int(float(s))}"
    except Exception:
        return s


def load_table(candidates, label):
    path = first_existing(candidates)
    if path is None:
        print(f"[WARN] no {label} table found")
        return None
    print(f"[load {label}]", path)
    df = pd.read_csv(path, sep=None, engine="python")
    df.columns = [str(c).strip() for c in df.columns]
    if "topic" not in df.columns:
        df = df.rename(columns={df.columns[0]: "topic"})
    df["topic"] = df["topic"].map(normalize_topic_name)
    return df


def register_species_genome(cfg):
    chrom_sizes = cfg.get("chrom_sizes")
    if chrom_sizes is not None and Path(chrom_sizes).exists():
        genome = crested.Genome(str(cfg["genome_fasta"]), str(chrom_sizes))
    else:
        genome = crested.Genome(str(cfg["genome_fasta"]))
    crested.register_genome(genome)
    return genome


def attach_topic_annotation(adata, annot_df, qc_df):
    adata.obs["topic"] = [normalize_topic_name(x) for x in adata.obs_names]

    if annot_df is not None:
        annot = annot_df.copy().set_index("topic")
        for col in annot.columns:
            adata.obs[col] = adata.obs["topic"].map(annot[col])

    if qc_df is not None:
        qc = qc_df.copy().set_index("topic")
        for col in qc.columns:
            adata.obs[f"qc__{col}"] = adata.obs["topic"].map(qc[col])

    return adata


def add_predictions_to_layer(adata, model, layer_name=MODEL_LAYER):
    print("[predict] crested.tl.predict")
    predictions = crested.tl.predict(adata, model)
    predictions = np.asarray(predictions)

    # Tutorial: adata.layers["biccn_model"] = predictions.T
    if predictions.shape == (adata.n_vars, adata.n_obs):
        layer = predictions.T
    elif predictions.shape == (adata.n_obs, adata.n_vars):
        layer = predictions
    else:
        raise ValueError(
            f"Unexpected prediction shape {predictions.shape}; "
            f"expected {(adata.n_vars, adata.n_obs)} or {(adata.n_obs, adata.n_vars)}"
        )

    adata.layers[layer_name] = layer
    print(f"[layer] {layer_name}:", adata.layers[layer_name].shape)
    return layer


def make_combined_layer(adata, pred_layer=MODEL_LAYER, combined_layer=COMBINED_LAYER):
    # Tutorial: adata.layers['combined'] = (adata.X + adata.layers["biccn_model"]) / 2
    adata.layers[combined_layer] = (adata.X + adata.layers[pred_layer]) / 2
    print(f"[layer] {combined_layer}:", adata.layers[combined_layer].shape)


def save_sort_filter_qc_plot(adata, out_png):
    try:
        crested.pl.qc.sort_and_filter_cutoff(
            adata,
            model_name=COMBINED_LAYER,
            cutoffs=[500, 1000, 2000],
            max_k=5000,
        )
        plt.savefig(out_png, dpi=220, bbox_inches="tight")
        plt.close()
        print("[save fig]", out_png)
    except Exception as e:
        print("[WARN] sort_and_filter_cutoff plot failed:", type(e).__name__, e)


def export_filtered_region_table(adata_filtered, out_tsv):
    df = adata_filtered.var.copy()
    df.index.name = "region"
    df.to_csv(out_tsv, sep="\t")
    print("[save]", out_tsv)


def export_class_annotation(adata, out_tsv):
    df = adata.obs.copy()
    df.index.name = "class_name"
    df.to_csv(out_tsv, sep="\t")
    print("[save]", out_tsv)


In [4]:
def normalize_topic_name(x):
    x = str(x).strip()
    if x.startswith("Topic"):
        return x
    try:
        return f"Topic{int(float(x))}"
    except Exception:
        return x


def split_celltypes(x):
    if pd.isna(x):
        return []
    return [i.strip() for i in str(x).split(",") if i.strip()]


def build_topic_to_celltypes(annot_df, label_col="final_class"):
    topic_to_cts = {}

    for _, row in annot_df.iterrows():
        topic = normalize_topic_name(row["topic"])

        if "is_general" in annot_df.columns:
            if str(row["is_general"]).lower() == "true":
                continue

        cts = split_celltypes(row[label_col])
        if len(cts) > 0:
            topic_to_cts[topic] = cts

    return topic_to_cts

In [5]:
def aggregate_layer_topic_to_celltype(
    adata, annot_df, layer_name="combined", label_col="final_class", agg="mean"
):
    topic_to_cts = build_topic_to_celltypes(annot_df, label_col=label_col)

    topic_names = [normalize_topic_name(x) for x in adata.obs_names]
    all_cts = sorted(set(ct for cts in topic_to_cts.values() for ct in cts))

    X = np.asarray(adata.layers[layer_name])

    rows = []
    used_cts = []

    for ct in all_cts:
        idx = [
            i for i, topic in enumerate(topic_names)
            if ct in topic_to_cts.get(topic, [])
        ]

        if len(idx) == 0:
            continue

        sub = X[idx, :]

        if agg == "max":
            vec = sub.max(axis=0)
        else:
            vec = sub.mean(axis=0)

        rows.append(vec)
        used_cts.append(ct)

    X_ct = np.vstack(rows).astype(np.float32)

    adata_ct = ad.AnnData(
        X=X_ct,
        obs=pd.DataFrame(index=used_cts),
        var=adata.var.copy(),
    )

    adata_ct.layers[layer_name] = X_ct

    print("[celltype adata]", adata_ct)
    print("[celltypes]", used_cts)

    return adata_ct

In [6]:
def gini_1d(x):
    x = np.asarray(x, dtype=np.float64)
    x = np.abs(x)

    if np.all(x == 0):
        return 0.0

    x = np.sort(x)
    n = len(x)
    return (2 * np.sum((np.arange(1, n + 1)) * x) / (n * np.sum(x))) - (n + 1) / n

def plot_celltype_gini_tutorial_style(
    adata_ct,
    layer_name,
    out_png,
    max_rank=5000,
    cutoffs=(500, 1000, 2000),
    smooth_window=100,
):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    X = np.asarray(adata_ct.layers[layer_name])
    celltypes = list(adata_ct.obs_names)

    # 每个 region 在 celltype 维度上的 Gini
    region_gini = np.apply_along_axis(gini_1d, 0, X)

    plt.figure(figsize=(12, 7))

    for i, ct in enumerate(celltypes):
        # 每个 celltype 自己按 score 排序
        order = np.argsort(X[i, :])[::-1][:max_rank]

        y_raw = region_gini[order]

        # 关键：rolling smooth，不然就是毛线
        y = (
            pd.Series(y_raw)
            .rolling(window=smooth_window, min_periods=1, center=True)
            .mean()
            .values
        )

        x = np.arange(1, len(order) + 1)
        plt.plot(x, y, linewidth=1.5, label=ct)

    for c in cutoffs:
        plt.axvline(c, color="black", linestyle="--", linewidth=1.2)

    plt.xlabel("Ranked regions per cell type")
    plt.ylabel("Gini score across cell types")
    plt.title("Cell-type-level specificity cutoff")

    plt.ylim(0, 1)

    plt.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=7,
        frameon=False,
    )

    plt.savefig(out_png, dpi=220, bbox_inches="tight")
    plt.close()

    print("[save]", out_png)

In [7]:
tag = "human"
cfg = DATASETS[tag]

adata = ad.read_h5ad(cfg["adata"])
annot_df = load_table(cfg["annotation_candidates"], "annotation")

model = keras.models.load_model(cfg["model"])

# prediction
if "model_prediction" not in adata.layers:
    pred = crested.tl.predict(
        input=adata,
        model=model,
        genome=cfg["genome_fasta"],
        batch_size=4,
    )
    print("[pred shape]", pred.shape)
    print("[adata shape]", adata.shape)

    if pred.shape == adata.shape:
        adata.layers["model_prediction"] = pred
    elif pred.T.shape == adata.shape:
        adata.layers["model_prediction"] = pred.T
    else:
        raise ValueError(f"Prediction shape {pred.shape} does not match adata {adata.shape}")

# combined
adata.layers["combined"] = (
    np.asarray(adata.X) + np.asarray(adata.layers["model_prediction"])
) / 2

#  topic → celltype
adata_ct = aggregate_layer_topic_to_celltype(
    adata,
    annot_df,
    layer_name="combined",
    label_col="final_class",
)

plot_celltype_gini_tutorial_style(
    adata_ct,
    layer_name="combined",
    out_png="human_celltype_gini.png",
    max_rank=5000,
    cutoffs=(500, 1000, 2000),
    smooth_window=100,
)

[load annotation] /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/data/3_topic_annotation_human_pycistopic.tsv
2026-04-29T14:12:53.317471+0200 INFO Lazily importing module crested.tl. This could take a second...
103852/103852 ━━━━━━━━━━━━━━━━━━━━ 1473s 14ms/step
[pred shape] (415405, 100)
[adata shape] (100, 415405)
[celltype adata] AnnData object with n_obs × n_vars = 17 × 415405
    var: 'n_classes', 'chr', 'start', 'end', 'split'
    layers: 'combined'
[celltypes] ['CGE_LGE_progenitors', 'CGE_interneuron', 'ExIPC', 'ExNeu_IT', 'ExNeu_Non_IT', 'LGE_FOXP1_ISL1_MSN', 'LGE_FOXP1_PENK_MSN', 'LGE_FOXP2_TSHZ1_MSN', 'LGE_OB_interneuron', 'MGE_interneuron', 'MGE_progenitors', 'Microglia', 'Neuroblast', 'OPC', 'TriIPC_Astrocyte', 'Vascular', 'dorsal_RG']
[save] human_celltype_gini.png


In [10]:
def run_one_species(tag, cfg):
    print(f"\n===== {tag} =====")
    species_out = OUTDIR / tag
    species_out.mkdir(parents=True, exist_ok=True)

    modisco_out = species_out / f"modisco_results_{tag}_top{TOP_K}"
    modisco_out.mkdir(parents=True, exist_ok=True)

    print("[load adata]", cfg["adata"])
    adata = ad.read_h5ad(cfg["adata"])
    print(adata)

    annot_df = load_table(cfg["annotation_candidates"], "annotation")
    qc_df = load_table(cfg["qc_candidates"], "QC")
    attach_topic_annotation(adata, annot_df, qc_df)
    export_class_annotation(adata, species_out / f"{tag}_class_annotation_used.tsv")

    print("[register genome]")
    register_species_genome(cfg)

    print("[load model]", cfg["model"])
    model = crested.utils.load_model(str(cfg["model"]))
    print(model)

    # Tutorial step 1: predictions for all regions
    add_predictions_to_layer(adata, model, MODEL_LAYER)

    # Tutorial step 2: combined = ground truth + prediction
    make_combined_layer(adata, MODEL_LAYER, COMBINED_LAYER)

    # Tutorial QC plot for choosing top_k
    save_sort_filter_qc_plot(
        adata,
        species_out / f"{tag}_sort_and_filter_cutoff_combined.png",
    )

    # Tutorial step 3: select most informative/specific regions per class
    print(f"[filter] top_k={TOP_K}, method={SPECIFICITY_METHOD}, model_name={COMBINED_LAYER}")
    adata_filtered = crested.pp.sort_and_filter_regions_on_specificity(
        adata,
        model_name=COMBINED_LAYER,
        top_k=TOP_K,
        method=SPECIFICITY_METHOD,
        inplace=False,
    )
    print(adata_filtered)

    export_filtered_region_table(
        adata_filtered,
        species_out / f"{tag}_filtered_regions_top{TOP_K}_{SPECIFICITY_METHOD}.tsv",
    )

    filtered_h5ad = species_out / f"{tag}_adata_filtered_top{TOP_K}_{SPECIFICITY_METHOD}.h5ad"
    adata_filtered.write_h5ad(filtered_h5ad)
    print("[save]", filtered_h5ad)

    # Tutorial step 4: contribution scores per class
    if RUN_CONTRIBUTION:
        print("[contribution_scores_specific] all classes, target_idx=None")
        crested.tl.contribution_scores_specific(
            input=adata_filtered,
            target_idx=None,
            model=model,
            output_dir=str(modisco_out),
            method=CONTRIB_METHOD,
        )

    # Tutorial step 5: TF-MoDISco-lite
    if RUN_TFMODISCO:
        print("[motif db] crested.get_motif_db()")
        try:
            meme_db, motif_to_tf_file = crested.get_motif_db()
            print("[meme_db]", meme_db)
            print("[motif_to_tf_file]", motif_to_tf_file)
        except Exception as e:
            print("[WARN] crested.get_motif_db failed; running without meme_db")
            print(type(e).__name__, e)
            meme_db, motif_to_tf_file = None, None

        print("[tfmodisco]")
        kwargs = dict(
            window=MODISCO_WINDOW,
            output_dir=str(modisco_out),
            contrib_dir=str(modisco_out),
            report=REPORT,
            max_seqlets=MAX_SEQLETS,
        )
        if meme_db is not None:
            kwargs["meme_db"] = meme_db

        crested.tl.modisco.tfmodisco(**kwargs)

    return adata, adata_filtered


stage2_results = {}
for tag in RUN_SPECIES:
    stage2_results[tag] = run_one_species(tag, DATASETS[tag])



===== human =====
[load adata] /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/human_topics.h5ad
AnnData object with n_obs × n_vars = 100 × 415405
    obs: 'file_path', 'n_open_regions'
    var: 'n_classes', 'chr', 'start', 'end', 'split'
[load annotation] /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/data/3_topic_annotation_human_pycistopic.tsv
[load QC] /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/data/3_topic_qc_metrics_human.tsv
[save] /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage2_crested_enhancer_code/human/human_class_annotation_used.tsv
[register genome]
2026-04-28T14:30:02.436919+0200 INFO Genome hg38 registered.
[load model] /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/deeptopic_human/final_model.keras
<Functional name=functional, built=True>
[predict] crested.tl.predict
3246/3246 ━━━━━━━━━━━━━━━━━━━━ 234s 72ms/step
[layer] model_prediction: (100, 415405)
[layer] combined: (100, 415405)
2026-04-28T14:34:15.413265+0200 INFO

/tmp/ipykernel_1426682/1516923208.py:101: UserWarning: constrained_layout not applied because axes sizes collapsed to zero.  Try making figure larger or Axes decorations smaller.
  plt.savefig(out_png, dpi=220, bbox_inches="tight")


[save fig] /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage2_crested_enhancer_code/human/human_sort_and_filter_cutoff_combined.png
[filter] top_k=2000, method=gini, model_name=combined
2026-04-28T14:34:41.816986+0200 INFO After sorting and filtering, kept 200000 regions.


/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/miniconda3/envs/work/lib/python3.11/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 100 × 200000
    obs: 'file_path', 'n_open_regions', 'topic', 'final_class', 'Ratio_cells_in_topic', 'Ratio_group_in_population', 'is_general', 'qc__Log10_Assignments', 'qc__Assignments', 'qc__Cells_in_binarized_topic', 'qc__Coherence', 'qc__Marginal_topic_dist', 'qc__Gini_index'
    var: 'n_classes', 'chr', 'start', 'end', 'split', 'Class name', 'rank', 'gini_score'
    layers: 'model_prediction', 'combined'
[save] /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage2_crested_enhancer_code/human/human_filtered_regions_top2000_gini.tsv
[save] /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage2_crested_enhancer_code/human/human_adata_filtered_top2000_gini.h5ad
[contribution_scores_specific] all classes, target_idx=None
2026-04-28T14:34:50.216781+0200 INFO Calculating contribution scores for 1 class(es) and 2000 region(s).


...

2026-04-28T15:58:24.098358+0200 INFO Calculating contribution scores for 1 class(es) and 2000 region(s).


Model: 100%|██████████| 1/1 [02:32<00:00, 152.71s/it]


2026-04-28T16:01:00.990997+0200 INFO Calculating contribution scores for 1 class(es) and 2000 region(s).


Model:   0%|          | 0/1 [01:49<?, ?it/s]


KeyboardInterrupt: 

## Notes

- Start with `RUN_SPECIES = ["human"]`.
- If it is too slow, set `TOP_K = 500`.
- If `tomtom` is unavailable, keep `REPORT = False`.
- The folder `modisco_results_<species>_top<TOP_K>` is the key output for downstream motif pattern analysis.
